# Original Flat PPO C + Post-Validation (Config A protocol)
**Original cells are preserved in their initial sequence.** In `POSTVAL_ONLY=True` mode, training cells and legacy cells that could overwrite `final_results` are conditionally gated. New `PV-*` cells are placed below all 37 original cells.
- Attempt 1 = Y trong CSV Flat C; attempts 2–3: T=0.9/1.1; checkpoint 140/seed.
- Does not delete or overwrite legacy data. Each execution creates a new results directory.
- Run All on Colab GPU; original code runs initialization only, no retraining.


In [ ]:
POSTVAL_ONLY = True
assert POSTVAL_ONLY, 'This notebook is only for Post-Val; training is not executed in this version.'
print('POSTVAL_ONLY — training/old result writers disabled')

POSTVAL_ONLY — training/old result writers disabled


# Flat PPO — Config C — Matched Compute — T4 GPU

**Purpose:** controlled flat/mixed PPO baseline for the curriculum analysis.

This notebook is derived from the original Config C training setup. The PPO configuration,
reward functions, model, tokenizer, and generation settings are kept unchanged.

**Only the curriculum mechanism is removed:**
- all Easy + Medium + Hard training samples are pooled from the start;
- no Phase 1 / Phase 2 / Phase 3 training;
- the reward vector is fixed to Config C's **Phase 3** weights;
- the total number of PPO updates is matched to the **actual** Phase 1 + Phase 2 + Phase 3
  updates already completed by each Config C seed.

Run on **NVIDIA T4** only.

## 1. Install, mount Drive, and verify T4

In [ ]:
!pip install -q "trl==0.11.4" transformers sentence-transformers accelerate pandas scipy matplotlib

from google.colab import drive
drive.mount('/content/drive')

import os, time, math, random, logging, json, re, glob, gc
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger("Flat-PPO")

assert torch.cuda.is_available(), "CUDA GPU is required."
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
assert "T4" in GPU_NAME.upper(), f"Expected NVIDIA T4, but found: {GPU_NAME}"

DEVICE = "cuda"
print("CUDA device:", torch.cuda.current_device())
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.8 MB/s eta 0:00:00
Mounted at /content/drive
GPU: Tesla T4
CUDA device: 0
VRAM (GB): 14.56


## 2. Paths and seeds

In [ ]:
DATA_DIR = "/content/drive/MyDrive/Data_Khoaluan/03_stratified_benchmark"
T5_SFT_DIR = "/content/drive/MyDrive/Data_Khoaluan/T5_base"

# Existing Config C is READ-ONLY: only its diagnostics are used to determine matched compute.
CONFIG_C_DIR = "/content/drive/MyDrive/Data_Khoaluan/ppo_config_c_v1"

# New output directory: this notebook never writes into RL_configa.
FLAT_OUTPUT_DIR = "/content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1"
os.makedirs(FLAT_OUTPUT_DIR, exist_ok=True)

SEED_LIST = [42, 123, 777]

print("DATA_DIR      =", DATA_DIR)
print("T5_SFT_DIR    =", T5_SFT_DIR)
print("CONFIG_C_DIR  =", CONFIG_C_DIR)
print("FLAT_OUTPUT   =", FLAT_OUTPUT_DIR)
print("SEEDS         =", SEED_LIST)

DATA_DIR      = /content/drive/MyDrive/Data_Khoaluan/03_stratified_benchmark
T5_SFT_DIR    = /content/drive/MyDrive/Data_Khoaluan/T5_base
CONFIG_C_DIR  = /content/drive/MyDrive/Data_Khoaluan/ppo_config_c_v1
FLAT_OUTPUT   = /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1
SEEDS         = [42, 123, 777]


## 3. Load the new split

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

train_toxic = train_df.reset_index(drop=True)
val_toxic   = val_df.reset_index(drop=True)
test_toxic  = test_df.reset_index(drop=True)

print("train =", len(train_toxic))
print("val   =", len(val_toxic))
print("test  =", len(test_toxic))
print("\nTraining band distribution:")
print(train_toxic["band"].value_counts())

assert len(train_toxic) == 2424, f"Expected 2424 filtered training rows, got {len(train_toxic)}"
assert set(train_toxic["band"].str.lower().unique()) == {"easy", "medium", "hard"}

train = 2424
val   = 672
test  = 1312

Training band distribution:
band
easy      1149
medium     754
hard       521
Name: count, dtype: int64


## 4. Load the same reward functions/models as Config C

In [ ]:
from transformers import T5TokenizerFast, AutoTokenizer, AutoModelForSequenceClassification
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F

tokenizer = T5TokenizerFast.from_pretrained(T5_SFT_DIR)
TASK_PREFIX = 'detoxify: '
MAX_SOURCE_LEN = 256
MAX_NEW_TOKENS = 128

def to_list(x):
    if torch.is_tensor(x): return x.detach().cpu().tolist()
    if isinstance(x, np.ndarray): return x.tolist()
    return list(x)

tox_tokenizer = AutoTokenizer.from_pretrained('unitary/unbiased-toxic-roberta')
tox_model = AutoModelForSequenceClassification.from_pretrained('unitary/unbiased-toxic-roberta').to(DEVICE)
tox_model.eval()
id2label = tox_model.config.id2label
TOX_IDX = next((int(i) for i, l in id2label.items() if str(l).lower() in {'toxic','toxicity'}), 0)

@torch.no_grad()
def compute_r_tox(texts):
    enc = tox_tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=256).to(DEVICE)
    probs = torch.sigmoid(tox_model(**enc).logits.float())[:, TOX_IDX]
    return to_list(1.0 - probs)

sbert_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=DEVICE)

def compute_r_sem(X_texts, Y_texts):
    eX = sbert_model.encode(X_texts, convert_to_tensor=True, device=DEVICE)
    eY = sbert_model.encode(Y_texts, convert_to_tensor=True, device=DEVICE)
    return to_list((F.cosine_similarity(eX, eY, dim=1) + 1.0) / 2.0)

gpt2_tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE)
gpt2_model.eval()
MAX_LOSS_CLAMP = 20.0

@torch.no_grad()
def compute_perplexity(text):
    enc = gpt2_tokenizer(text, return_tensors='pt', truncation=True, max_length=256).to(DEVICE)
    if enc.input_ids.size(1) < 2:
        return 1.0
    loss = torch.clamp(gpt2_model(enc.input_ids, labels=enc.input_ids).loss, max=MAX_LOSS_CLAMP)
    return torch.exp(loss).item()

def compute_r_flu(texts):
    """Standard version — sentence by sentence, identical to Table 2. No batching."""
    return [1.0 / (1.0 + math.log(max(compute_perplexity(t), 1.0 + 1e-6)) / 5.0) for t in texts]

def compute_r_len(X_texts, Y_texts):
    rewards = []
    for x, y in zip(X_texts, Y_texts):
        x_len, y_len = max(len(x.split()), 1), len(y.split())
        ratio = y_len / x_len
        if 0.7 <= ratio <= 1.3: r = 1.0
        elif ratio < 0.7: r = y_len / (0.7 * x_len)
        else: r = (1.3 * x_len) / max(y_len, 1)
        rewards.append(r)
    return rewards

POLITENESS_LEXICON = [
    'please', 'could you', 'would you', 'might you', 'can you', 'will you',
    'hi', 'hello', 'hey', 'greetings', 'howdy', 'good morning', 'good evening', 'good afternoon',
    'goodbye', 'bye', 'see you later', 'thank', 'grateful', 'gratitude',
    'sorry', 'woops', 'oops', 'whoops', 'apologize',
    'is okay', 'not worry', 'no big deal', 'not a big deal', 'no problem', 'no worries',
    'is fine', 'you are good', "it's fine", "it's okay", 'let me know', 'for you',
]
P_SIZE = len(POLITENESS_LEXICON)

def compute_r_pol(Y_texts):
    return [min(sum(1 for w in POLITENESS_LEXICON if w in y.lower()) / P_SIZE, 1.0) for y in Y_texts]

def compute_r_total(X_texts, Y_texts, weights):
    r_tox, r_sem, r_flu = compute_r_tox(Y_texts), compute_r_sem(X_texts, Y_texts), compute_r_flu(Y_texts)
    r_len, r_pol = compute_r_len(X_texts, Y_texts), compute_r_pol(Y_texts)
    total = [weights['tox']*r_tox[i] + weights['sem']*r_sem[i] + weights['flu']*r_flu[i]
             + weights['len']*r_len[i] + weights['pol']*r_pol[i] for i in range(len(Y_texts))]
    return total, {'r_tox':r_tox,'r_sem':r_sem,'r_flu':r_flu,'r_len':r_len,'r_pol':r_pol}

logger.info('Reward functions (standard, non-batched) ready.')

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## 5. Same PPO configuration as Config C

In [ ]:
from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead, create_reference_model

BETA_KL, CLIP_EPS, GAE_GAMMA, GAE_LAMBDA = 0.05, 0.2, 0.99, 0.95
LEARNING_RATE, PPO_BATCH_SIZE, PPO_MINI_BATCH_SIZE = 1.4e-5, 8, 4

ppo_config = PPOConfig(
    learning_rate=LEARNING_RATE, batch_size=PPO_BATCH_SIZE, mini_batch_size=PPO_MINI_BATCH_SIZE,
    ppo_epochs=4, init_kl_coef=BETA_KL, cliprange=CLIP_EPS, cliprange_value=CLIP_EPS,
    gamma=GAE_GAMMA, lam=GAE_LAMBDA,
)
logger.info('PPOConfig ready.')

/usr/local/lib/python3.13/dist-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(


## 6. Flat reward and matched-compute definition

In [ ]:
# Config C Phase-3 reward is FIXED for the entire flat run.
FLAT_WEIGHTS = {
    "tox": 0.15,
    "sem": 0.30,
    "flu": 0.20,
    "len": 0.25,
    "pol": 0.10,
}

MAX_NEW_TOKENS = 128
CHECKPOINT_EVERY = 50

def actual_config_c_steps(seed):
    phase_paths = [
        os.path.join(CONFIG_C_DIR, f"seed_{seed}", "diagnostics_phase1_easy.csv"),
        os.path.join(CONFIG_C_DIR, f"seed_{seed}", "diagnostics_phase2_balance.csv"),
        os.path.join(CONFIG_C_DIR, f"seed_{seed}", "diagnostics_phase3_refinement.csv"),
    ]

    counts = []
    for p in phase_paths:
        if not os.path.exists(p):
            print(f" Warning: Missing file {p} -> Hardcoding step = 50.")
            counts.append(50)
        else:
            try:
                df = pd.read_csv(p)
                if "step" not in df.columns:
                    counts.append(50)
                else:
                    counts.append(len(df))
            except Exception:
                counts.append(50)

    return counts, sum(counts)

for seed in SEED_LIST:
    phase_counts, total = actual_config_c_steps(seed)
    print(f"Seed {seed}: Phase1={phase_counts[0]}, Phase2={phase_counts[1]}, "
          f"Phase3={phase_counts[2]} -> Flat target={total} updates")

print("\nFlat reward:", FLAT_WEIGHTS)
print("No Easy/Medium/Hard reward switching occurs during Flat PPO.")

Seed 42: Phase1=40, Phase2=40, Phase3=60 -> Flat target=140 updates
Seed 123: Phase1=40, Phase2=40, Phase3=60 -> Flat target=140 updates
Seed 777: Phase1=40, Phase2=40, Phase3=60 -> Flat target=140 updates

Flat reward: {'tox': 0.15, 'sem': 0.3, 'flu': 0.2, 'len': 0.25, 'pol': 0.1}
No Easy/Medium/Hard reward switching occurs during Flat PPO.


## 7. Flat PPO training loop

In [ ]:
from transformers import T5TokenizerFast
from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead, create_reference_model

def safe_generate_flat(ppo_trainer, query_tensors, max_attempts=5):
    """
    Uses the ORIGINAL Config-A generation settings.
    If a response is empty/all-padding, regenerate the affected batch rather than
    changing generation parameters.
    """
    last_responses = None

    for attempt in range(max_attempts):
        response_tensors = ppo_trainer.generate(
            query_tensors,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            top_k=0,
            top_p=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )
        last_responses = response_tensors

        bad = []
        for j, r in enumerate(response_tensors):
            if r.numel() == 0:
                bad.append(j)
                continue
            # Remove padding only for the validity check; generation config is unchanged.
            valid = r[r != tokenizer.pad_token_id]
            if valid.numel() == 0:
                bad.append(j)

        if not bad:
            return response_tensors

        logger.warning(
            f"Empty/all-padding response detected (attempt {attempt+1}/{max_attempts}); "
            "regenerating with the same generation configuration."
        )

    raise RuntimeError(
        "Could not obtain a non-empty PPO response after retries. "
        "Training stopped rather than silently changing the generation configuration."
    )

def run_flat_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    seed_dir = os.path.join(FLAT_OUTPUT_DIR, f"seed_{seed}")
    os.makedirs(seed_dir, exist_ok=True)

    # Match the actual number of PPO updates performed by Config C for this seed.
    phase_counts, target_steps = actual_config_c_steps(seed)

    # Full training corpus: Easy + Medium + Hard pooled from the beginning.
    texts_all = train_toxic["text"].astype(str).tolist()
    assert len(texts_all) == 2424

    # Resume-safe: continue from a saved flat checkpoint if available.
    resume_candidates = sorted(
        glob.glob(os.path.join(seed_dir, "flat_step_*")),
        key=lambda p: int(re.search(r"flat_step_(\d+)$", p).group(1))
    )
    if resume_candidates:
        resume_dir = resume_candidates[-1]
        m = re.search(r"flat_step_(\d+)$", resume_dir)
        start_step = int(m.group(1))
        policy_dir = resume_dir
        print(f"[Seed {seed}] Resuming from {policy_dir} at step {start_step}/{target_steps}")
    else:
        start_step = 0
        policy_dir = T5_SFT_DIR
        print(f"[Seed {seed}] Starting from Full-SFT checkpoint.")

    policy = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(policy_dir).to(DEVICE)
    ref = create_reference_model(policy).to(DEVICE)
    trainer = PPOTrainer(config=ppo_config, model=policy, ref_model=ref, tokenizer=tokenizer)

    diagnostics_path = os.path.join(seed_dir, "flat_diagnostics.csv")
    diagnostics = []
    if os.path.exists(diagnostics_path):
        old = pd.read_csv(diagnostics_path)
        diagnostics = old.to_dict("records")

    step = start_step
    t0 = time.time()

    while step < target_steps:
        idx = random.sample(range(len(texts_all)), PPO_BATCH_SIZE)
        X_batch = [texts_all[i] for i in idx]

        query_tensors = [
            tokenizer(
                TASK_PREFIX + x,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_SOURCE_LEN
            ).input_ids[0].to(DEVICE)
            for x in X_batch
        ]

        response_tensors = safe_generate_flat(trainer, query_tensors)
        Y_batch = [tokenizer.decode(r, skip_special_tokens=True) for r in response_tensors]

        rewards_list, _ = compute_r_total(X_batch, Y_batch, FLAT_WEIGHTS)
        reward_tensors = [torch.tensor(r, dtype=torch.float32) for r in rewards_list]

        stats = trainer.step(query_tensors, response_tensors, reward_tensors)
        step += 1

        row = {
            "step": step,
            "mean_reward": float(np.mean(rewards_list)),
            "kl": stats.get("objective/kl"),
            "policy_loss": stats.get("ppo/loss/policy"),
        }
        diagnostics.append(row)

        if step % 5 == 0:
            print(
                f"[Seed {seed}] step={step}/{target_steps} "
                f"R={row['mean_reward']:.4f} KL={row['kl']} "
                f"policy_loss={row['policy_loss']}"
            )

        if step % CHECKPOINT_EVERY == 0 or step == target_steps:
            ckpt_dir = os.path.join(seed_dir, f"flat_step_{step}")
            policy.save_pretrained(ckpt_dir)
            tokenizer.save_pretrained(ckpt_dir)
            pd.DataFrame(diagnostics).to_csv(diagnostics_path, index=False)
            print(f"[Seed {seed}] checkpoint saved: {ckpt_dir}")

    # Final manifest for this seed
    manifest = {
        "seed": seed,
        "target_steps": target_steps,
        "config_a_phase_steps": {
            "phase1_easy": phase_counts[0],
            "phase2_balance": phase_counts[1],
            "phase3_refinement": phase_counts[2],
        },
        "train_rows": len(texts_all),
        "reward_weights": FLAT_WEIGHTS,
        "generation": {
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": True,
            "top_k": 0,
            "top_p": 1.0,
            "pad_token_id": tokenizer.pad_token_id,
        },
        "ppo": {
            "learning_rate": LEARNING_RATE,
            "batch_size": PPO_BATCH_SIZE,
            "mini_batch_size": PPO_MINI_BATCH_SIZE,
            "ppo_epochs": 4,
            "kl_coef": BETA_KL,
            "cliprange": CLIP_EPS,
            "gamma": GAE_GAMMA,
            "gae_lambda": GAE_LAMBDA,
        },
        "flat_definition": "Easy+Medium+Hard pooled from the beginning; no curriculum phases.",
    }
    with open(os.path.join(seed_dir, "manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)

    del trainer, ref, policy
    gc.collect()
    torch.cuda.empty_cache()

    print(f"[Seed {seed}] DONE — {step} updates.")
    return step

## 8. Run all three flat seeds

In [ ]:
# [ORIGINAL FLAT C — preserved, disabled in POSTVAL_ONLY]
if not POSTVAL_ONLY:
    for seed in SEED_LIST:
        run_flat_seed(seed)
else:
    print('SKIP original training/evaluation writer cell 16 (POSTVAL_ONLY)')

SKIP original training/evaluation writer cell 16 (POSTVAL_ONLY)


## 9. Final verification

In [ ]:
# [ORIGINAL FLAT C — preserved, disabled in POSTVAL_ONLY]
if not POSTVAL_ONLY:
    print("===== FLAT PPO FINAL CHECK =====")
    for seed in SEED_LIST:
        seed_dir = os.path.join(FLAT_OUTPUT_DIR, f"seed_{seed}")
        phase_counts, target = actual_config_c_steps(seed)
        diag = os.path.join(seed_dir, "flat_diagnostics.csv")

        if os.path.exists(diag):
            df = pd.read_csv(diag)
            actual = len(df)
            print(
                f"Seed {seed}: flat={actual} updates | "
                f"target={target} | Config C phases={phase_counts}"
            )
            assert actual == target, f"Seed {seed}: update mismatch"
        else:
            print(f"Seed {seed}: diagnostics missing")

    print("\nOutput directory:", FLAT_OUTPUT_DIR)
else:
    print('SKIP original training/evaluation writer cell 18 (POSTVAL_ONLY)')

SKIP original training/evaluation writer cell 18 (POSTVAL_ONLY)


## Important design choice

The original Config C uses different reward vectors across its curriculum phases.  
For the **flat/mixed control**, the reward vector is fixed to **Config C Phase 3**:

\[
(\lambda_{\mathrm{tox}},\lambda_{\mathrm{sem}},\lambda_{\mathrm{flu}},
\lambda_{\mathrm{len}},\lambda_{\mathrm{pol}})
=(0.30,0.30,0.20,0.10,0.10).
\]

This notebook therefore changes the **curriculum/data exposure schedule** while keeping the final Config-A reward configuration fixed. It does **not** assign Easy/Medium/Hard-specific rewards to individual samples.

The notebook also does not use early stopping for Flat PPO, because the control must match the actual total PPO updates of Config C for each seed.

##  RUN EVALUATION (INFERENCE) ON CHECKPOINTS

In [ ]:
TAU_REDETECT, SEM_THRESHOLD, FLU_THRESHOLD, MAX_REGEN_ATTEMPTS = 0.30, 0.70, 0.70, 3
TEMPERATURES = [0.7, 0.9, 1.1]

def generate_one_with_temp(model, x_text, temperature):
    inputs = tokenizer(TASK_PREFIX+x_text, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN).to(DEVICE)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                                   temperature=temperature, top_k=0, top_p=1.0)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

def passes_post_validation(x_text, y_text):
    t_out = 1.0 - compute_r_tox([y_text])[0]
    r_sem = compute_r_sem([x_text], [y_text])[0]
    r_flu = compute_r_flu([y_text])[0]
    y_len, x_len = len(y_text.split()), max(len(x_text.split()), 1)
    ratio = y_len / x_len
    return (t_out < TAU_REDETECT) and (r_sem > SEM_THRESHOLD) and (r_flu > FLU_THRESHOLD) and (0.7 <= ratio <= 1.3)

def generate_with_post_validation(model, X_texts):
    """Standard version — sentence by sentence, identical to Table 2. No batching."""
    Y_out, n_failed = [], 0
    for x in X_texts:
        passed, last_y = False, None
        for attempt in range(MAX_REGEN_ATTEMPTS):
            y = generate_one_with_temp(model, x, TEMPERATURES[attempt])
            last_y = y
            if passes_post_validation(x, y):
                passed = True
                break
        if not passed:
            n_failed += 1
        Y_out.append(last_y)
    return Y_out, n_failed

def evaluate_checkpoint(model, eval_df, use_post_validation=False):
    X_texts, labels, bands = eval_df['text'].tolist(), eval_df['label'].tolist(), eval_df['band'].tolist()
    if use_post_validation:
        Y_texts, n_failed = generate_with_post_validation(model, X_texts)
    else:
        Y_texts, n_failed = generate_batch_deterministic(model, X_texts), None
    STA = compute_r_tox(Y_texts)
    SIM = compute_r_sem(X_texts, Y_texts)
    FLUE = compute_r_flu(Y_texts)
    Joint = [(s*si*f)**(1/3) for s,si,f in zip(STA,SIM,FLUE)]
    df = pd.DataFrame({'X':X_texts,'Y':Y_texts,'label':labels,'band':bands,'STA':STA,'Sim':SIM,'Flu':FLUE,'Joint':Joint})
    return df, n_failed

def summarize(df):
    return {'STA':float(df['STA'].mean()),'Sim':float(df['Sim'].mean()),'Flu':float(df['Flu'].mean()),'Joint':float(df['Joint'].mean())}

logger.info('Post-Validation (standard) + evaluate_checkpoint/summarize ready.')

In [ ]:
# [ORIGINAL FLAT C — preserved, disabled in POSTVAL_ONLY]
if not POSTVAL_ONLY:
    EVAL_BATCH_SIZE = 16
    TAU_REDETECT, SEM_THRESHOLD, FLU_THRESHOLD, MAX_REGEN_ATTEMPTS = 0.30, 0.70, 0.70, 3
    TEMPERATURES = [0.7, 0.9, 1.1]

    def generate_batch_deterministic(model, texts, batch_size=EVAL_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS):
        results = []
        n_batches = math.ceil(len(texts) / batch_size)
        for bi in range(n_batches):
            batch_texts = texts[bi*batch_size:(bi+1)*batch_size]
            inputs = [TASK_PREFIX + t for t in batch_texts]
            try:
                enc = tokenizer(inputs, return_tensors='pt', padding=True, truncation=True, max_length=MAX_SOURCE_LEN).to(DEVICE)
                with torch.no_grad():
                    out_ids = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, num_beams=4)
                results.extend([tokenizer.decode(o, skip_special_tokens=True) for o in out_ids])
            except RuntimeError as e:
                logger.error(f'Loi batch {bi}: {e}. Thu tung cau.')
                for t in batch_texts:
                    try:
                        enc1 = tokenizer(TASK_PREFIX+t, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN).to(DEVICE)
                        with torch.no_grad():
                            o = model.generate(**enc1, max_new_tokens=max_new_tokens, do_sample=False, num_beams=4)
                        results.append(tokenizer.decode(o[0], skip_special_tokens=True))
                    except Exception as e2:
                        logger.error(f'Fail 1 cau: {e2}')
                        results.append(t)
        return results

    def generate_one_with_temp(model, x_text, temperature):
        inputs = tokenizer(TASK_PREFIX+x_text, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN).to(DEVICE)
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                                       temperature=temperature, top_k=0, top_p=1.0)
        return tokenizer.decode(out_ids[0], skip_special_tokens=True)

    def passes_post_validation(x_text, y_text):
        t_out = 1.0 - compute_r_tox([y_text])[0]
        r_sem = compute_r_sem([x_text], [y_text])[0]
        r_flu = compute_r_flu([y_text])[0]
        y_len, x_len = len(y_text.split()), max(len(x_text.split()), 1)
        ratio = y_len / x_len
        return (t_out < TAU_REDETECT) and (r_sem > SEM_THRESHOLD) and (r_flu > FLU_THRESHOLD) and (0.7 <= ratio <= 1.3)

    def generate_with_post_validation(model, X_texts):
        Y_out, n_failed = [], 0
        for x in X_texts:
            passed, last_y = False, None
            for attempt in range(MAX_REGEN_ATTEMPTS):
                y = generate_one_with_temp(model, x, TEMPERATURES[attempt])
                last_y = y
                if passes_post_validation(x, y):
                    passed = True
                    break
            if not passed:
                n_failed += 1
            Y_out.append(last_y)
        return Y_out, n_failed

    def evaluate_checkpoint(model, eval_df, use_post_validation=False):
        X_texts, labels, bands = eval_df['text'].tolist(), eval_df['label'].tolist(), eval_df['band'].tolist()
        if use_post_validation:
            Y_texts, n_failed = generate_with_post_validation(model, X_texts)
        else:
            Y_texts, n_failed = generate_batch_deterministic(model, X_texts), None
        STA  = compute_r_tox(Y_texts)
        SIM  = compute_r_sem(X_texts, Y_texts)
        FLUE = compute_r_flu(Y_texts)
        Joint = [(s*si*f)**(1/3) for s, si, f in zip(STA, SIM, FLUE)]
        df = pd.DataFrame({'X':X_texts,'Y':Y_texts,'label':labels,'band':bands,
                           'STA':STA,'Sim':SIM,'Flu':FLUE,'Joint':Joint})
        return df, n_failed

    def summarize(df):
        return {'STA':float(df['STA'].mean()),'Sim':float(df['Sim'].mean()),
                'Flu':float(df['Flu'].mean()),'Joint':float(df['Joint'].mean())}

    logger.info('generate_batch_deterministic + evaluate_checkpoint + summarize san sang.')

    import gc

    RESULTS_DIR = os.path.join(FLAT_OUTPUT_DIR, 'final_results')
    os.makedirs(RESULTS_DIR, exist_ok=True)

    print("=== EVALUATION OF FLAT PPO CHECKPOINTS ===")

    for seed in SEED_LIST:
        print(f"\n--- Evaluating Seed {seed} ---")
        out_csv = os.path.join(RESULTS_DIR, f'per_sample_seed{seed}_phase_flat.csv')

        if os.path.exists(out_csv):
            print(f" Results already exist for Seed {seed} at {out_csv}, skipping generation.")
            continue

        seed_dir = os.path.join(FLAT_OUTPUT_DIR, f"seed_{seed}")

        # Find latest checkpoint (highest step)
        import glob, re
        ckpt_dirs = glob.glob(os.path.join(seed_dir, "flat_step_*"))
        if not ckpt_dirs:
            print(f" No checkpoint found for Seed {seed}")
            continue

        best_ckpt = sorted(ckpt_dirs, key=lambda p: int(re.search(r'flat_step_(\d+)', p).group(1)))[-1]
        print(f"Loading checkpoint: {best_ckpt}")

        m_flat = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(best_ckpt).to(DEVICE)
        m_flat.eval()

        print("Starting generation and evaluation...")
        per_sample_df, _ = evaluate_checkpoint(m_flat, test_toxic, use_post_validation=False)
        per_sample_df.to_csv(out_csv, index=False)
        print(f" Saved Seed {seed} results successfully.")

        del m_flat
        gc.collect()
        torch.cuda.empty_cache()

    print("\n COMPLETED EVALUATION (INFERENCE) FOR 3 SEEDS!")
else:
    print('SKIP original training/evaluation writer cell 22 (POSTVAL_ONLY)')

SKIP original training/evaluation writer cell 22 (POSTVAL_ONLY)


##  SUMMARY OF EVALUATION RESULTS (MEAN ± STD)

In [ ]:
# [ORIGINAL FLAT C — preserved, disabled in POSTVAL_ONLY]
if not POSTVAL_ONLY:
    import pandas as pd
    import numpy as np
    import os

    RESULTS_DIR = os.path.join(FLAT_OUTPUT_DIR, 'final_results')
    # Create directory if not exists
    os.makedirs(RESULTS_DIR, exist_ok=True)

    SEED_LIST = [42, 123, 777]

    def summarize(df):
        return {
            'STA': float(df['STA'].mean()),
            'Sim': float(df['Sim'].mean()),
            'Flu': float(df['Flu'].mean()),
            'Joint': float(df['Joint'].mean())
        }

    records = []

    for seed in SEED_LIST:
        # Read Phase FLAT results
        csv_path = os.path.join(RESULTS_DIR, f'per_sample_seed{seed}_phase_flat.csv')
        if not os.path.exists(csv_path):
            print(f" Evaluation file {csv_path} not found. Please run the evaluation cells above first!")
            continue

        df = pd.read_csv(csv_path)

        # Compute for 3 groups
        full_metrics = summarize(df)
        toxic_metrics = summarize(df[df['label'] == 'toxic'])
        nontoxic_metrics = summarize(df[df['label'] == 'nontoxic'])

        records.append({
            'Seed': seed,
            'Group': 'FULL',
            'STA': full_metrics['STA'], 'Sim': full_metrics['Sim'], 'Flu': full_metrics['Flu'], 'Joint': full_metrics['Joint']
        })
        records.append({
            'Seed': seed,
            'Group': 'TOXIC',
            'STA': toxic_metrics['STA'], 'Sim': toxic_metrics['Sim'], 'Flu': toxic_metrics['Flu'], 'Joint': toxic_metrics['Joint']
        })
        records.append({
            'Seed': seed,
            'Group': 'NONTOXIC',
            'STA': nontoxic_metrics['STA'], 'Sim': nontoxic_metrics['Sim'], 'Flu': nontoxic_metrics['Flu'], 'Joint': nontoxic_metrics['Joint']
        })

    if len(records) > 0:
        results_df = pd.DataFrame(records)
        print(" PER-SEED RESULTS (PHASE FLAT):")
        display(results_df.round(4))

        # Compute Mean and Std Dev across 3 seeds
        summary_df = results_df.groupby('Group').agg(['mean', 'std']).reset_index()

        # Format cleanly as Mean ± Std
        formatted_records = []
        for _, row in summary_df.iterrows():
            grp = row['Group'].values[0] if isinstance(row['Group'], pd.Series) else row['Group'][0]
            formatted_records.append({
                'Group': grp,
                'STA': f"{row[('STA','mean')]:.3f} ± {row[('STA','std')]:.3f}",
                'Sim': f"{row[('Sim','mean')]:.3f} ± {row[('Sim','std')]:.3f}",
                'Flu': f"{row[('Flu','mean')]:.3f} ± {row[('Flu','std')]:.3f}",
                'Joint': f"{row[('Joint','mean')]:.3f} ± {row[('Joint','std')]:.3f}",
            })

        final_table = pd.DataFrame(formatted_records)
        # Sort order: FULL -> TOXIC -> NONTOXIC
        final_table['Group'] = pd.Categorical(final_table['Group'], categories=['FULL', 'TOXIC', 'NONTOXIC'], ordered=True)
        final_table = final_table.sort_values('Group').reset_index(drop=True)

        print("\n SUMMARY TABLE MEAN ± STD (FLAT PPO):")
        display(final_table)
    else:
        print(" No data available for aggregation.")
else:
    print('SKIP original training/evaluation writer cell 24 (POSTVAL_ONLY)')

SKIP original training/evaluation writer cell 24 (POSTVAL_ONLY)


##  BEHAVIORAL METRICS AND PPL (IDENTICAL TO CONFIG A)

In [ ]:
# [ORIGINAL FLAT C — preserved, disabled in POSTVAL_ONLY]
if not POSTVAL_ONLY:
    import pandas as pd
    import numpy as np
    import os
    import math

    def tokenize_simple(text): return str(text).lower().split()

    def levenshtein_token(a, b):
        n, m = len(a), len(b)
        dp = [[0]*(m+1) for _ in range(n+1)]
        for i in range(n+1): dp[i][0] = i
        for j in range(m+1): dp[0][j] = j
        for i in range(1, n+1):
            for j in range(1, m+1):
                dp[i][j] = dp[i-1][j-1] if a[i-1]==b[j-1] else 1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])
        return dp[n][m]

    def compute_edit_behavior(row):
        x, y = str(row['X']).strip(), str(row['Y']).strip()
        xt, yt = tokenize_simple(x), tokenize_simple(y)
        xl = max(len(xt),1); yl = len(yt)
        return pd.Series({'unchanged': x.lower()==y.lower(), 'token_edit_rate': levenshtein_token(xt,yt)/xl,
                            'truncated': yl<xl, 'clause_loss': ((xl-yl)/xl) > 0.30})

    _all = []
    RESULTS_DIR = os.path.join(FLAT_OUTPUT_DIR, 'final_results')
    SEED_LIST = [42, 123, 777]

    # 1. PER-SEED TABLE
    print('=== 1. BEHAVIORAL METRICS AND PPL PER SEED ===')
    seed_results = []
    seed_ppl = []

    for seed in SEED_LIST:
        p = os.path.join(RESULTS_DIR, f'per_sample_seed{seed}_phase_flat.csv')
        if os.path.exists(p):
            df = pd.read_csv(p)
            df['seed'] = seed
            df['phase'] = 'phase_flat'
            _all.append(df)

            # Compute behavioral metrics
            m = df.apply(compute_edit_behavior, axis=1)
            seed_results.append({'Seed':seed, 'n':len(df),
                              'Unchanged (%)':round(m['unchanged'].mean()*100,1),
                              'Token Edit Rate (%)':round(m['token_edit_rate'].mean()*100,1),
                              'Truncation (%)':round(m['truncated'].mean()*100,1),
                              'Clause Loss >30% (%)':round(m['clause_loss'].mean()*100,1)})

            # Compute PPL
            ppls = pd.Series([compute_perplexity(str(y)) for y in df['Y']])
            pct = ((ppls>=20)&(ppls<=150)).mean()
            seed_ppl.append({'Seed':seed, 'mean_PPL':round(ppls.mean(),1), 'trong[20,150]':round(pct*100,1)})

    if not _all:
        print(" No evaluation CSV files found in final_results.")
    else:
        print("\n[Behavioral Metrics Per Seed - FULL]")
        display(pd.DataFrame(seed_results))
        print("\n[PPL Per Seed - FULL]")
        display(pd.DataFrame(seed_ppl))

        # 2. POOLED TABLE SIMILAR TO CONFIG A
        all_per_sample_pooled = pd.concat(_all, ignore_index=True)

        print('\n\n=== 2. POOLED TABLE SIMILAR TO CONFIG A ===')

        print('\n[TABLE OF 4 BEHAVIORAL METRICS — FULL]')
        results = []
        m = all_per_sample_pooled.apply(compute_edit_behavior, axis=1)
        results.append({'Phase':'phase_flat (Pooled)','n':len(all_per_sample_pooled),
                          'Unchanged Rate (%)':round(m['unchanged'].mean()*100,1),
                          'Token Edit Rate (%)':round(m['token_edit_rate'].mean()*100,1),
                          'Truncation Rate (%)':round(m['truncated'].mean()*100,1),
                          'Clause Loss >30% (%)':round(m['clause_loss'].mean()*100,1)})
        edit_behavior_full = pd.DataFrame(results)
        print(edit_behavior_full.to_string(index=False))
        edit_behavior_full.to_csv(os.path.join(RESULTS_DIR, 'edit_behavior_full.csv'), index=False)

        print('\n[TABLE OF 4 BEHAVIORAL METRICS — Toxic-only]')
        results_toxic = []
        _pooled_toxic_behavior = all_per_sample_pooled[all_per_sample_pooled['label']=='toxic']
        m = _pooled_toxic_behavior.apply(compute_edit_behavior, axis=1)
        results_toxic.append({'Phase':'phase_flat (Pooled)','n':len(_pooled_toxic_behavior),
                          'Unchanged Rate (%)':round(m['unchanged'].mean()*100,1),
                          'Token Edit Rate (%)':round(m['token_edit_rate'].mean()*100,1),
                          'Truncation Rate (%)':round(m['truncated'].mean()*100,1),
                          'Clause Loss >30% (%)':round(m['clause_loss'].mean()*100,1)})
        edit_behavior_toxic = pd.DataFrame(results_toxic)
        print(edit_behavior_toxic.to_string(index=False))
        edit_behavior_toxic.to_csv(os.path.join(RESULTS_DIR, 'edit_behavior_toxic_only.csv'), index=False)

        print('\n[TABLE OF PPL — FULL]')
        ppl_rows = []
        ppls = pd.Series([compute_perplexity(str(y)) for y in all_per_sample_pooled['Y']])
        pct = ((ppls>=20)&(ppls<=150)).mean()
        ppl_rows.append({'phase':'phase_flat (Pooled)','mean_PPL':round(ppls.mean(),1),'median_PPL':round(ppls.median(),1),'pct_in_20_150':round(pct,3)})
        print(f"phase_flat (Pooled): mean={ppls.mean():7.1f}, trong[20,150]={pct:.1%}")
        ppl_full = pd.DataFrame(ppl_rows)
        ppl_full.to_csv(os.path.join(RESULTS_DIR, 'ppl_full.csv'), index=False)

        print('\n[TABLE OF PPL — Toxic-only]')
        ppl_rows_toxic = []
        _pooled_toxic = all_per_sample_pooled[all_per_sample_pooled['label']=='toxic']
        ppls = pd.Series([compute_perplexity(str(y)) for y in _pooled_toxic['Y']])
        pct = ((ppls>=20)&(ppls<=150)).mean()
        ppl_rows_toxic.append({'phase':'phase_flat (Pooled)','mean_PPL':round(ppls.mean(),1),'median_PPL':round(ppls.median(),1),'pct_in_20_150':round(pct,3)})
        print(f"phase_flat (Pooled): mean={ppls.mean():7.1f}, trong[20,150]={pct:.1%}")
        ppl_toxic = pd.DataFrame(ppl_rows_toxic)
        ppl_toxic.to_csv(os.path.join(RESULTS_DIR, 'ppl_toxic_only.csv'), index=False)
else:
    print('SKIP original training/evaluation writer cell 26 (POSTVAL_ONLY)')

SKIP original training/evaluation writer cell 26 (POSTVAL_ONLY)


## Step E1 — Evaluate: Load checkpoints & generate outputs (inference only, no training)

In [ ]:
# [ORIGINAL FLAT C — preserved, disabled in POSTVAL_ONLY]
if not POSTVAL_ONLY:
    import gc, glob, re
    from trl import AutoModelForSeq2SeqLMWithValueHead

    RESULTS_DIR = os.path.join(FLAT_OUTPUT_DIR, 'final_results')
    os.makedirs(RESULTS_DIR, exist_ok=True)

    # Baseline SFT
    BASELINE_CSV = os.path.join(RESULTS_DIR, 'baseline_per_sample.csv')
    if os.path.exists(BASELINE_CSV):
        print("Baseline already exists, loading...")
        baseline_per_sample = pd.read_csv(BASELINE_CSV)
    else:
        print("Generating baseline (SFT)...")
        baseline_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(T5_SFT_DIR).to(DEVICE)
        baseline_model.eval()
        baseline_per_sample, _ = evaluate_checkpoint(baseline_model, test_toxic, use_post_validation=False)
        baseline_per_sample.to_csv(BASELINE_CSV, index=False)
        del baseline_model
        gc.collect()
        torch.cuda.empty_cache()

    baseline_metrics_full     = summarize(baseline_per_sample)
    baseline_metrics_toxic    = summarize(baseline_per_sample[baseline_per_sample['label']=='toxic'])
    baseline_metrics_nontoxic = summarize(baseline_per_sample[baseline_per_sample['label']=='nontoxic'])
    print(f"Baseline | FULL Joint={baseline_metrics_full['Joint']:.3f} | TOXIC Joint={baseline_metrics_toxic['Joint']:.3f} | NONTOXIC Joint={baseline_metrics_nontoxic['Joint']:.3f}")

    # PPO Flat checkpoints
    for seed in SEED_LIST:
        out_csv = os.path.join(RESULTS_DIR, f'per_sample_seed{seed}_phase_flat.csv')
        if os.path.exists(out_csv):
            print(f"[Seed {seed}] Flat result already exists, skipping.")
            continue

        seed_dir = os.path.join(FLAT_OUTPUT_DIR, f"seed_{seed}")
        ckpt_dirs = sorted(glob.glob(os.path.join(seed_dir, "flat_step_*")),
                           key=lambda p: int(re.search(r'flat_step_(\d+)', p).group(1)))
        if not ckpt_dirs:
            print(f"[Seed {seed}] No checkpoint found! Skipping.")
            continue

        best_ckpt = ckpt_dirs[-1]
        print(f"[Seed {seed}] Loading: {best_ckpt}")
        m = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(best_ckpt).to(DEVICE)
        m.eval()
        df, _ = evaluate_checkpoint(m, test_toxic, use_post_validation=False)
        df.to_csv(out_csv, index=False)
        print(f"[Seed {seed}] Saved: {out_csv}")
        del m
        gc.collect()
        torch.cuda.empty_cache()

    print("\nAll inference done!")
else:
    print('SKIP original training/evaluation writer cell 28 (POSTVAL_ONLY)')

SKIP original training/evaluation writer cell 28 (POSTVAL_ONLY)


## Step E2 — Per-seed tables: table3/3b/3c + table4/4b/4c (FULL / Toxic / Non-toxic)

In [ ]:
# [ORIGINAL FLAT C — preserved, disabled in POSTVAL_ONLY]
if not POSTVAL_ONLY:
    def summarize(df):
        return {'STA':float(df['STA'].mean()),'Sim':float(df['Sim'].mean()),'Flu':float(df['Flu'].mean()),'Joint':float(df['Joint'].mean())}

    METRICS = ['STA', 'Sim', 'Flu', 'Joint']

    for seed in SEED_LIST:
        fname = f'per_sample_seed{seed}_phase_flat.csv'
        fpath = os.path.join(RESULTS_DIR, fname)
        if not os.path.exists(fpath):
            print(f"[Seed {seed}] Missing {fpath}, skipping.")
            continue

        df = pd.read_csv(fpath)

        t3  = [{'Phase': 'Baseline (SFT)', **baseline_metrics_full},
                {'Phase': 'Flat PPO',      **summarize(df)}]
        t3b = [{'Phase': 'Baseline (SFT)', **baseline_metrics_toxic},
                {'Phase': 'Flat PPO',      **summarize(df[df['label']=='toxic'])}]
        t3c = [{'Phase': 'Baseline (SFT)', **baseline_metrics_nontoxic},
                {'Phase': 'Flat PPO',      **summarize(df[df['label']=='nontoxic'])}]

        # Add Gain column
        def add_gain(rows, base_joint):
            for r in rows:
                r['Gain'] = 0.0 if r['Phase']=='Baseline (SFT)' else round((r['Joint']-base_joint)/base_joint*100, 2)
            return rows

        t3  = add_gain(t3,  baseline_metrics_full['Joint'])
        t3b = add_gain(t3b, baseline_metrics_toxic['Joint'])
        t3c = add_gain(t3c, baseline_metrics_nontoxic['Joint'])

        pd.DataFrame(t3).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3_seed{seed}.csv'), index=False)
        pd.DataFrame(t3b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3b_seed{seed}.csv'), index=False)
        pd.DataFrame(t3c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3c_seed{seed}.csv'), index=False)

        print(f'=== SEED {seed} --- TABLE 3 (FULL) ===')
        print(pd.DataFrame(t3).round(3).to_string(index=False))
        print(f'\n=== SEED {seed} --- TABLE 3b (Toxic) ===')
        print(pd.DataFrame(t3b).round(3).to_string(index=False))
        print(f'\n=== SEED {seed} --- TABLE 3c (Non-toxic) ===')
        print(pd.DataFrame(t3c).round(3).to_string(index=False))

        # Table 4/4b/4c by difficulty band
        t4, t4b, t4c = [], [], []
        for band in ['easy', 'medium', 'hard']:
            bf = df[df['band']==band]
            bt = df[(df['band']==band)&(df['label']=='toxic')]
            bn = df[(df['band']==band)&(df['label']=='nontoxic')]
            if len(bf)>0: t4.append({'Difficulty': band.capitalize(), 'Count': len(bf), **summarize(bf)})
            if len(bt)>0: t4b.append({'Difficulty': band.capitalize(), 'Count': len(bt), **summarize(bt)})
            if len(bn)>0: t4c.append({'Difficulty': band.capitalize(), 'Count': len(bn), **summarize(bn)})

        t4.append({'Difficulty':'Overall','Count':len(df), **summarize(df)})
        t4b.append({'Difficulty':'Overall','Count':len(df[df['label']=='toxic']), **summarize(df[df['label']=='toxic'])})
        t4c.append({'Difficulty':'Overall','Count':len(df[df['label']=='nontoxic']), **summarize(df[df['label']=='nontoxic'])})

        pd.DataFrame(t4).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4_seed{seed}.csv'), index=False)
        pd.DataFrame(t4b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4b_seed{seed}.csv'), index=False)
        pd.DataFrame(t4c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4c_seed{seed}.csv'), index=False)

        print(f'\n=== SEED {seed} --- TABLE 4 (FULL, band) ===')
        print(pd.DataFrame(t4).round(3).to_string(index=False))
        print(f'\n=== SEED {seed} --- TABLE 4b (Toxic, band) ===')
        print(pd.DataFrame(t4b).round(3).to_string(index=False))
        print(f'\n=== SEED {seed} --- TABLE 4c (Non-toxic, band) ===')
        print(pd.DataFrame(t4c).round(3).to_string(index=False))
        print()
else:
    print('SKIP original training/evaluation writer cell 30 (POSTVAL_ONLY)')

SKIP original training/evaluation writer cell 30 (POSTVAL_ONLY)


## Step E3 — Aggregate: Mean+/-Std across seeds + Bootstrap CI

In [ ]:
# [ORIGINAL FLAT C — preserved, disabled in POSTVAL_ONLY]
if not POSTVAL_ONLY:
    import numpy as np

    METRICS = ['STA', 'Sim', 'Flu', 'Joint']

    # ========== Mean+-Std across seeds ==========
    def print_main_table(prefix, index_col='Phase'):
        dfs = {}
        for seed in SEED_LIST:
            path = os.path.join(RESULTS_DIR, f'{prefix}_seed{seed}.csv')
            if os.path.exists(path):
                dfs[seed] = pd.read_csv(path).set_index(index_col)
        if not dfs:
            print(f'Khong tim thay file {prefix}_seed*.csv'); return
        keys = list(dfs[SEED_LIST[0]].index)
        for k in keys:
            print(f'--- {k} ---')
            for metric in METRICS:
                values, used_seeds = [], []
                for seed in SEED_LIST:
                    if seed in dfs and k in dfs[seed].index and metric in dfs[seed].columns:
                        values.append(dfs[seed].loc[k, metric]); used_seeds.append(seed)
                if not values: continue
                seed_str = ' | '.join(f'S{s}={v:.3f}' for s, v in zip(used_seeds, values))
                mean_v = np.mean(values)
                std_v  = np.std(values, ddof=1) if len(values) > 1 else 0.0
                print(f'  {metric:10s}: {seed_str}  ->  Mean+-Std = {mean_v:.4f} +- {std_v:.4f}')
        print()

    for prefix, idx_col in [('table3','Phase'),('table3b','Phase'),('table3c','Phase'),
                              ('table4','Difficulty'),('table4b','Difficulty'),('table4c','Difficulty')]:
        sep = '='*80
        print(sep)
        print(f'BANG CHINH --- {prefix.upper()} (FLAT PPO, Mean+-Std across seeds)')
        print(sep)
        print_main_table(prefix, idx_col)

    # ========== Bootstrap CI ==========
    def bootstrap_ci(values, n_boot=1000, ci=0.95, seed=42):
        rng = np.random.default_rng(seed)
        values = np.asarray(values)
        boot_means = np.array([rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_boot)])
        lo = np.percentile(boot_means, (1-ci)/2*100)
        hi = np.percentile(boot_means, (1+ci)/2*100)
        return values.mean(), lo, hi

    _all = []
    for seed in SEED_LIST:
        p = os.path.join(RESULTS_DIR, f'per_sample_seed{seed}_phase_flat.csv')
        if os.path.exists(p):
            df = pd.read_csv(p)
            df['seed'] = seed
            df['phase'] = 'Flat PPO'
            _all.append(df)

    if _all:
        all_pooled = pd.concat(_all, ignore_index=True)
        print(f'Total pooled rows: {len(all_pooled)}')

        def build_ci_table(df, label_filter=None):
            sub = df if label_filter is None else df[df['label']==label_filter]
            rows = []
            for g in sub['phase'].unique():
                gdf = sub[sub['phase']==g]
                row = {'phase': g, 'n_pooled': len(gdf)}
                for col in METRICS:
                    if col not in gdf.columns: continue
                    mean_v, lo, hi = bootstrap_ci(gdf[col].values)
                    row[col] = round(mean_v, 4)
                    row[f'{col}_CI95'] = f'[{lo:.3f}, {hi:.3f}]'
                rows.append(row)
            return pd.DataFrame(rows)

        for label, filt in [('FULL', None), ('TOXIC', 'toxic'), ('NONTOXIC', 'nontoxic')]:
            ci_df = build_ci_table(all_pooled, filt)
            ci_df.to_csv(os.path.join(RESULTS_DIR, f'bootstrap_ci_{label.lower()}.csv'), index=False)
            print(f'\n=== Bootstrap CI --- {label} ===')
            print(ci_df.to_string(index=False))
else:
    print('SKIP original training/evaluation writer cell 32 (POSTVAL_ONLY)')

SKIP original training/evaluation writer cell 32 (POSTVAL_ONLY)


## Step E4 — Edit Behaviour + PPL (FULL / Toxic / Non-toxic, Pooled)

In [ ]:
# [ORIGINAL FLAT C — preserved, disabled in POSTVAL_ONLY]
if not POSTVAL_ONLY:
    # Edit behaviour
    def tokenize_simple(text): return str(text).lower().split()

    def levenshtein_token(a, b):
        n, m = len(a), len(b)
        dp = [[0]*(m+1) for _ in range(n+1)]
        for i in range(n+1): dp[i][0] = i
        for j in range(m+1): dp[0][j] = j
        for i in range(1, n+1):
            for j in range(1, m+1):
                dp[i][j] = dp[i-1][j-1] if a[i-1]==b[j-1] else 1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])
        return dp[n][m]

    def compute_edit_behavior(row):
        x, y = str(row['X']).strip(), str(row['Y']).strip()
        xt, yt = tokenize_simple(x), tokenize_simple(y)
        xl = max(len(xt),1); yl = len(yt)
        return pd.Series({'unchanged': x.lower()==y.lower(),
                          'token_edit_rate': levenshtein_token(xt,yt)/xl,
                          'truncated': yl<xl,
                          'clause_loss': ((xl-yl)/xl) > 0.30})

    def behaviour_table(df, label='FULL'):
        if label == 'TOXIC':    sub = df[df['label']=='toxic']
        elif label == 'NONTOXIC': sub = df[df['label']=='nontoxic']
        else:                     sub = df
        m = sub.apply(compute_edit_behavior, axis=1)
        return pd.DataFrame([{
            'Phase': 'Flat PPO', 'n': len(sub),
            'Unchanged (%)': round(m['unchanged'].mean()*100, 1),
            'Token changed (%)': round(m['token_edit_rate'].mean()*100, 1),
            'Truncated (%)': round(m['truncated'].mean()*100, 1),
            'Clause loss >30% (%)': round(m['clause_loss'].mean()*100, 1)
        }])

    def ppl_table(df, label='FULL'):
        if label == 'TOXIC':    sub = df[df['label']=='toxic']
        elif label == 'NONTOXIC': sub = df[df['label']=='nontoxic']
        else:                     sub = df
        ppls = pd.Series([compute_perplexity(str(y)) for y in sub['Y']])
        pct  = ((ppls>=20)&(ppls<=150)).mean()
        return {'phase':'Flat PPO','mean_PPL':round(ppls.mean(),1),
                'median_PPL':round(ppls.median(),1),'pct_in_20_150':round(pct,3)}

    if _all:
        for label in ['FULL', 'TOXIC', 'NONTOXIC']:
            if label == 'TOXIC':   sub_all = all_pooled[all_pooled['label']=='toxic']
            elif label=='NONTOXIC': sub_all = all_pooled[all_pooled['label']=='nontoxic']
            else:                   sub_all = all_pooled

            beh = behaviour_table(all_pooled, label)
            beh.to_csv(os.path.join(RESULTS_DIR, f'edit_behavior_{label.lower()}.csv'), index=False)
            print(f'\n=== BANG HANH VI --- {label} ===')
            print(beh.to_string(index=False))

            p = ppl_table(all_pooled, label)
            print(f'\n=== BANG PPL --- {label} ===')
            print(f"Flat PPO: mean={p['mean_PPL']:.1f}, median={p['median_PPL']:.1f}, trong[20,150]={p['pct_in_20_150']:.1%}")
            pd.DataFrame([p]).to_csv(os.path.join(RESULTS_DIR, f'ppl_{label.lower()}.csv'), index=False)

        print("\nHOAN TAT tat ca cac bang!")
else:
    print('SKIP original training/evaluation writer cell 34 (POSTVAL_ONLY)')

SKIP original training/evaluation writer cell 34 (POSTVAL_ONLY)


##  Rebuild Tables with STA (indep.) Metric

In [ ]:
# [ORIGINAL FLAT C — preserved, disabled in POSTVAL_ONLY]
if not POSTVAL_ONLY:
    # =====================================================================================
    # INDEPENDENT STA — FLAT PPO
    # 1) Load s-nlp/roberta_toxicity_classifier, rescore STA (indep.) on per_sample_*.csv
    # 2) Rebuild table3/3b/3c + table4/4b/4c (add STA (indep.) column)
    # 3) Rebuild Bootstrap CI (FULL/TOXIC/NON-TOXIC) (add STA (indep.) column)
    # =====================================================================================

    import os, re, glob, numpy as np, pandas as pd, torch, gc
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    METRICS    = ['STA', 'STA (indep.)', 'Sim', 'Flu', 'Joint']

    # -------------------------------------------------------------------------------------
    # STEP 1 — Independent STA Scoring on CSV files
    # -------------------------------------------------------------------------------------
    print('=' * 90); print('STEP 1 — Independent STA Scoring (s-nlp/roberta_toxicity_classifier)'); print('=' * 90)
    indep_tokenizer = AutoTokenizer.from_pretrained('s-nlp/roberta_toxicity_classifier')
    indep_model = AutoModelForSequenceClassification.from_pretrained('s-nlp/roberta_toxicity_classifier').to(DEVICE)
    indep_model.eval()

    id2label_indep = indep_model.config.id2label
    TOX_IDX_INDEP = next((int(i) for i, l in id2label_indep.items() if str(l).lower() in {'toxic', 'toxicity'}), 1)

    @torch.no_grad()
    def compute_sta_indep(texts, batch_size=32):
        out = []
        for i in range(0, len(texts), batch_size):
            batch = [str(t) for t in texts[i:i + batch_size]]
            enc = indep_tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256).to(DEVICE)
            probs = torch.softmax(indep_model(**enc).logits.float(), dim=-1)[:, TOX_IDX_INDEP]
            out.extend((1.0 - probs).cpu().tolist())
        return out

    csv_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*_phase_flat.csv'))) \
              + sorted(glob.glob(os.path.join(RESULTS_DIR, 'baseline_per_sample.csv')))

    gap_rows = []
    for path in csv_files:
        df = pd.read_csv(path)
        if 'STA (indep.)' not in df.columns:
            df['STA (indep.)'] = compute_sta_indep(df['Y'].tolist())
            df.to_csv(path, index=False)

        fname = os.path.basename(path)
        m = re.search(r'seed(\d+)_phase_flat', fname)
        seed, phase = (m.group(1), 'Flat PPO') if m else ('baseline', 'Baseline (SFT)')
        gap_rows.append({'seed': seed, 'phase': phase, 'n': len(df),
                          'STA (orig.)': round(df['STA'].mean(), 4),
                          'STA (indep.)': round(df['STA (indep.)'].mean(), 4),
                          'gap': round(df['STA'].mean() - df['STA (indep.)'].mean(), 4)})

    sta_gap_table = pd.DataFrame(gap_rows).sort_values(['phase', 'seed'])
    sta_gap_table.to_csv(os.path.join(RESULTS_DIR, 'sta_independent_comparison.csv'), index=False)
    print("\n=== Gap Comparison ===")
    print(sta_gap_table.to_string(index=False))

    del indep_model
    gc.collect(); torch.cuda.empty_cache()

    # -------------------------------------------------------------------------------------
    # STEP 2 — Rebuild table3 / table4
    # -------------------------------------------------------------------------------------
    print('\n' + '=' * 90); print('STEP 2 — Rebuild table3/table4 (add STA indep)'); print('=' * 90)

    def summarize2(df):
        return {m: float(df[m].mean()) for m in METRICS if m in df.columns}

    BASELINE_CSV = os.path.join(RESULTS_DIR, 'baseline_per_sample.csv')
    baseline_per_sample = pd.read_csv(BASELINE_CSV)
    baseline_metrics_full     = summarize2(baseline_per_sample)
    baseline_metrics_toxic    = summarize2(baseline_per_sample[baseline_per_sample['label'] == 'toxic'])
    baseline_metrics_nontoxic = summarize2(baseline_per_sample[baseline_per_sample['label'] == 'nontoxic'])

    for SEED in SEED_LIST:
        fpath = os.path.join(RESULTS_DIR, f'per_sample_seed{SEED}_phase_flat.csv')
        if not os.path.exists(fpath): continue
        df = pd.read_csv(fpath)

        t3  = [{'Phase': 'Baseline (SFT)', **baseline_metrics_full},     {'Phase': 'Flat PPO', **summarize2(df)}]
        t3b = [{'Phase': 'Baseline (SFT)', **baseline_metrics_toxic},    {'Phase': 'Flat PPO', **summarize2(df[df['label'] == 'toxic'])}]
        t3c = [{'Phase': 'Baseline (SFT)', **baseline_metrics_nontoxic}, {'Phase': 'Flat PPO', **summarize2(df[df['label'] == 'nontoxic'])}]

        pd.DataFrame(t3).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3_seed{SEED}.csv'), index=False)
        pd.DataFrame(t3b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3b_seed{SEED}.csv'), index=False)
        pd.DataFrame(t3c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3c_seed{SEED}.csv'), index=False)

        t4, t4b, t4c = [], [], []
        for band in ['easy', 'medium', 'hard']:
            bf, bt, bn = df[df['band']==band], df[(df['band']==band)&(df['label']=='toxic')], df[(df['band']==band)&(df['label']=='nontoxic')]
            if len(bf) > 0: t4.append({'Difficulty': band.capitalize(), 'Count': len(bf), **summarize2(bf)})
            if len(bt) > 0: t4b.append({'Difficulty': band.capitalize(), 'Count': len(bt), **summarize2(bt)})
            if len(bn) > 0: t4c.append({'Difficulty': band.capitalize(), 'Count': len(bn), **summarize2(bn)})

        t4.append({'Difficulty': 'Overall', 'Count': len(df), **summarize2(df)})
        t4b.append({'Difficulty': 'Overall', 'Count': len(df[df['label']=='toxic']), **summarize2(df[df['label']=='toxic'])})
        t4c.append({'Difficulty': 'Overall', 'Count': len(df[df['label']=='nontoxic']), **summarize2(df[df['label']=='nontoxic'])})

        pd.DataFrame(t4).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4_seed{SEED}.csv'), index=False)
        pd.DataFrame(t4b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4b_seed{SEED}.csv'), index=False)
        pd.DataFrame(t4c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4c_seed{SEED}.csv'), index=False)

    def print_main_table2(prefix, index_col='Phase'):
        dfs = {S: pd.read_csv(os.path.join(RESULTS_DIR, f'{prefix}_seed{S}.csv')).set_index(index_col) for S in SEED_LIST if os.path.exists(os.path.join(RESULTS_DIR, f'{prefix}_seed{S}.csv'))}
        if not dfs: return
        for k in dfs[SEED_LIST[0]].index:
            print(f'--- {k} ---')
            for metric in METRICS:
                vals = [dfs[S].loc[k, metric] for S in SEED_LIST if S in dfs and metric in dfs[S].columns]
                if not vals: continue
                print(f'  {metric:14s}: ' + ' | '.join(f'S{S}={v:.3f}' for S, v in zip([s for s in SEED_LIST if s in dfs], vals)) + f'  ->  Mean±Std = {np.mean(vals):.4f} ± {(np.std(vals, ddof=1) if len(vals)>1 else 0.0):.4f}')
        print()

    for prefix, idx_col in [('table3', 'Phase'), ('table3b', 'Phase'), ('table3c', 'Phase'),
                              ('table4', 'Difficulty'), ('table4b', 'Difficulty'), ('table4c', 'Difficulty')]:
        print('=' * 90); print(f'MAIN TABLE — {prefix} (with STA (indep.))'); print('=' * 90)
        print_main_table2(prefix, idx_col)

    # -------------------------------------------------------------------------------------
    # STEP 3 — Rebuild Bootstrap CI
    # -------------------------------------------------------------------------------------
    print('\n' + '=' * 90); print('STEP 3 — Rebuild Bootstrap CI (with STA indep)'); print('=' * 90)

    def bootstrap_ci(values, n_boot=1000, ci=0.95, seed=42):
        rng = np.random.default_rng(seed)
        values = np.asarray(values)
        boot_means = np.array([rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_boot)])
        lo, hi = np.percentile(boot_means, (1 - ci) / 2 * 100), np.percentile(boot_means, (1 + ci) / 2 * 100)
        return values.mean(), lo, hi

    _all = []
    for SEED in SEED_LIST:
        fpath = os.path.join(RESULTS_DIR, f'per_sample_seed{SEED}_phase_flat.csv')
        if os.path.exists(fpath):
            df = pd.read_csv(fpath)
            df['seed'], df['phase'] = SEED, 'Flat PPO'
            _all.append(df)
    all_per_sample_pooled = pd.concat(_all, ignore_index=True) if _all else None

    if all_per_sample_pooled is not None:
        for label, filt in [('FULL', None), ('TOXIC', 'toxic'), ('NON-TOXIC', 'nontoxic')]:
            sub = all_per_sample_pooled if filt is None else all_per_sample_pooled[all_per_sample_pooled['label'] == filt]
            rows = []
            for g in sub['phase'].unique():
                gdf = sub[sub['phase'] == g]
                row = {'phase': g, 'n_pooled': len(gdf)}
                for col in METRICS:
                    if col not in gdf.columns: continue
                    mean, lo, hi = bootstrap_ci(gdf[col].values)
                    row[col] = round(mean, 4); row[f'{col}_CI95'] = f'[{lo:.3f}, {hi:.3f}]'
                rows.append(row)
            ci_table = pd.DataFrame(rows)
            print(f'\n=== Bootstrap CI — {label} (with STA (indep.)) ===')
            print(ci_table.to_string(index=False))
            ci_table.to_csv(os.path.join(RESULTS_DIR, f'bootstrap_ci_{label.lower().replace("-", "")}.csv'), index=False)
else:
    print('SKIP original training/evaluation writer cell 36 (POSTVAL_ONLY)')

SKIP original training/evaluation writer cell 36 (POSTVAL_ONLY)


## PV-1 — Paths and configuration definitions (read-only, no overwriting)

In [ ]:
import os, gc, json, math, re, time
from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, GPT2TokenizerFast, GPT2LMHeadModel, T5TokenizerFast
from sentence_transformers import SentenceTransformer
from trl import AutoModelForSeq2SeqLMWithValueHead

ROOT = Path('/content/drive/MyDrive/Data_Khoaluan')
FLAT = ROOT/'flat_ppo_config_c_v1'
OLD_RESULTS = FLAT/'final_results'
SFT_CSV = OLD_RESULTS/'baseline_per_sample.csv'
SEEDS = [42,123,777]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TASK_PREFIX='detoxify: '
MAX_SOURCE_LEN=256
MAX_NEW_TOKENS=128
TAU_TOX=0.30
TAU_SEM=0.70
LEN_MIN,LEN_MAX=0.70,1.30
FLU_ORIG=0.70
FLU_CONV=1/(1+math.log(139.0)/5)
# To use exact previously documented 0.5033, change to FLU_CONV=0.5033
# NOTE: 1/(1+ln(139)/5) ~ 0.5033.
# MATCH CONFIG A: attempt 1 = saved beam-search Y, then T=.9 / 1.1.
REGEN_TEMPS=(0.9,1.1)
# Preserve original and create a NEW run directory on every normal launch.
RUN_DIR=None  # None -> timestamped NEW directory. Resume only by setting an explicit old run path.
RESUME_RUN=False
SCORER_VERSION='flatC_original_cell8_configA_independent_v3'
RUN_PROTOCOL_VERSION='flatC_configA_deterministic_first_v2'
if RUN_DIR is None:
    RUN_DIR=FLAT/'final_results_postval_fixed'/('run_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
else:
    RUN_DIR=Path(RUN_DIR)
if RUN_DIR.exists() and not RESUME_RUN:
    raise FileExistsError(f'Output run exists: {RUN_DIR}; set RESUME_RUN=True only for the same run')
RUN_DIR.mkdir(parents=True,exist_ok=True)
(RUN_DIR/'master_tables').mkdir(exist_ok=True)
print('Read-only:', FLAT)
print('New outputs:',RUN_DIR)
for seed in SEEDS:
    ck=FLAT/f'seed_{seed}'/'flat_step_140'
    csv=OLD_RESULTS/f'per_sample_seed{seed}_phase_flat.csv'
    for fp in [ck/'config.json',ck/'model.safetensors',csv]:
        if not fp.exists(): raise FileNotFoundError(fp)
if not SFT_CSV.exists(): raise FileNotFoundError(SFT_CSV)
if (RUN_DIR/'settings.json').exists():
    old_settings=json.loads((RUN_DIR/'settings.json').read_text())
    expected = {'seed_list': SEEDS, 'attempt_policy': 'deterministic_first',
                'regeneration_temperatures': list(REGEN_TEMPS), 'flu_original': FLU_ORIG,
                'flu_converted': FLU_CONV, 'tau_tox': TAU_TOX, 'tau_sem': TAU_SEM,
                'length_min': LEN_MIN, 'length_max': LEN_MAX,
                'scorer_version': SCORER_VERSION, 'protocol_version': RUN_PROTOCOL_VERSION}
    if old_settings != expected:
        raise ValueError('Resume refused: settings/scorer/protocol version changed. Start a NEW run.')
else:
  with open(RUN_DIR/'settings.json','w') as f:
      json.dump({'seed_list':SEEDS,'attempt_policy':'deterministic_first','regeneration_temperatures':REGEN_TEMPS,
               'flu_original':FLU_ORIG,'flu_converted':FLU_CONV,'tau_tox':TAU_TOX,'tau_sem':TAU_SEM,
               'length_min':LEN_MIN,'length_max':LEN_MAX,
               'scorer_version':SCORER_VERSION,'protocol_version':RUN_PROTOCOL_VERSION},f,indent=2)


Read-only: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1
New outputs: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/final_results_postval_fixed/run_20261014_133013


## PV-2 — Reuse original Flat C scorers; add independent STA

In [ ]:

# PV-2: toxicity, semantic, fluency, and PPL scorers from ORIGINAL FLAT C functions,
# does not reload tox/SBERT/GPT2 as a separate independent scoring system.
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from trl import AutoModelForSeq2SeqLMWithValueHead
assert all(callable(globals().get(k)) for k in
           ['compute_r_tox','compute_r_sem','compute_perplexity','compute_r_flu']), \
    'Run original Flat C initialization cells (cell 8) before running PV.'
t5tok = T5TokenizerFast.from_pretrained(T5_SFT_DIR)
_INDEP_MODEL = None
_INDEP_TOKENIZER = None
_INDEP_TOX_INDEX = None

def independent_sta(text):
    global _INDEP_MODEL, _INDEP_TOKENIZER, _INDEP_TOX_INDEX
    if _INDEP_MODEL is None:
        _INDEP_TOKENIZER = AutoTokenizer.from_pretrained('s-nlp/roberta_toxicity_classifier')
        _INDEP_MODEL = AutoModelForSequenceClassification.from_pretrained(
            's-nlp/roberta_toxicity_classifier').to(DEVICE).eval()
        _INDEP_TOX_INDEX = next((int(i) for i,l in _INDEP_MODEL.config.id2label.items()
                         if str(l).lower() in {'toxic','toxicity'}), 1)
    bt = _INDEP_TOKENIZER([str(text)], return_tensors='pt', padding=True,
                          truncation=True, max_length=256).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(_INDEP_MODEL(**bt).logits.float(),
                              dim=-1)[:,_INDEP_TOX_INDEX]
    return float(1.0-probs[0].item())

def assess(x,y):
    x,y=str(x),str(y)
    sta=float(compute_r_tox([y])[0])
    sim=float(compute_r_sem([x],[y])[0])
    ppl=float(compute_perplexity(y))
    flu=float(1/(1+math.log(max(ppl,1.000001))/5))
    ratio=len(y.split())/max(len(x.split()),1)
    tp=1-sta
    result={'STA':sta,'STA (indep.)':independent_sta(y),
            'Sim':sim,'Flu':flu,'PPL':ppl,'Joint':(max(sta,0)*max(sim,0)*max(flu,0))**(1/3),
            'length_ratio':ratio,
            'fail_toxic':int(tp>=TAU_TOX),
            'fail_semantic':int(sim<=TAU_SEM),
            'fail_length':int(not (LEN_MIN<=ratio<=LEN_MAX)),
            'fail_flu_original':int(not (flu>FLU_ORIG)),
            'fail_flu_converted':int(not (flu>FLU_CONV))}
    for rule in ('original','converted'):
        result['pass_'+rule]=int(not any(result[k] for k in
                         ['fail_toxic','fail_semantic','fail_length','fail_flu_'+rule]))
    return result
print('PV scorer = original Flat C reward/scoring functions + Config A independent STA.')


PV scorer = original Flat C reward/scoring functions + Config A independent STA.


## PV-3 — Preflight: check data and saved scores prior to execution

In [ ]:
# PREFLIGHT: validate original Flat C scorer columns before generating anything.
# The reference CSV might not contain all metrics: compare those present.
# This blocks mismatched evaluator settings rather than silently producing incomparable tables.
CHECK_N_PER_SEED = 8
ATOL_SCORE = 5e-4
for seed in SEEDS:
    ref = pd.read_csv(OLD_RESULTS/f'per_sample_seed{seed}_phase_flat.csv',keep_default_na=False)
    required = {'X','Y','label','band'}
    assert required.issubset(ref.columns), (seed, required-set(ref.columns))
    assert len(ref)==1312, (seed,len(ref))
    assert ref['label'].astype(str).str.strip().str.lower().eq('toxic').sum()==1150, seed
    inds=np.linspace(0,len(ref)-1,CHECK_N_PER_SEED,dtype=int)
    checked=[]
    for idx in inds:
        row=ref.iloc[int(idx)]
        new=assess(str(row.X),str(row.Y))
        # Compare only directly equivalent numeric columns present in saved Flat CSV.
        for col in ('STA','STA (indep.)','Sim','Flu','Joint','PPL'):
            if col not in ref.columns: continue
            old=pd.to_numeric(pd.Series([row[col]]),errors='coerce').iloc[0]
            if pd.isna(old): continue
            diff=abs(float(old)-float(new[col]))
            if diff>ATOL_SCORE:
                raise ValueError(
                    f'Preflight FAILED: seed={seed} row={idx} {col}: '
                    f'old={old}, recomputed={new[col]}, diff={diff}. '
                    'STOP: reconcile original Flat metric implementation before Post-Val.')
            checked.append(col)
    print(f'[PREFLIGHT] seed={seed}: verified {len(inds)} outputs; compared saved metric columns: {sorted(set(checked))}')
print('PREFLIGHT PASSED. Note: GPU runtime and checkpoint loading remain untested until executed.')


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: s-nlp/roberta_toxicity_classifier
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[PREFLIGHT] seed=42: verified 8 outputs; compared saved metric columns: ['Flu', 'Joint', 'STA', 'STA (indep.)', 'Sim']
[PREFLIGHT] seed=123: verified 8 outputs; compared saved metric columns: ['Flu', 'Joint', 'STA', 'STA (indep.)', 'Sim']
[PREFLIGHT] seed=777: verified 8 outputs; compared saved metric columns: ['Flu', 'Joint', 'STA', 'STA (indep.)', 'Sim']
PREFLIGHT PASSED. Note: GPU runtime and checkpoint loading remain untested until executed.


## PV-4 — Regenerate candidates, log per seed; checkpoint 140

In [ ]:
# MATCH CONFIG A: checkpoint-safe candidate log, shared between original/converted
import hashlib, random
def sha(text): return hashlib.sha256(str(text).encode('utf-8')).hexdigest()
def candidate_seed(seed,idx,attempt): return (int(seed)*1_000_003+int(idx))*10+int(attempt)
@torch.inference_mode()
def generate_one(model,x,temp,seed,idx,attempt):
    rs=candidate_seed(seed,idx,attempt)
    random.seed(rs); np.random.seed(rs%(2**32)); torch.manual_seed(rs)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(rs)
    enc=t5tok(TASK_PREFIX+str(x),return_tensors='pt',
              truncation=True,max_length=MAX_SOURCE_LEN).to(DEVICE)
    ids=model.generate(**enc,max_new_tokens=MAX_NEW_TOKENS,
                       do_sample=True,temperature=temp,top_k=0,top_p=1.0)
    return t5tok.decode(ids[0],skip_special_tokens=True)

def load_base(seed):
    path=OLD_RESULTS/f'per_sample_seed{seed}_phase_flat.csv'
    d=pd.read_csv(path,keep_default_na=False).reset_index(drop=True)
    required={'X','Y','label','band'}
    assert required.issubset(d.columns),(path,required-set(d.columns))
    assert not d[['X','Y']].isna().any().any()
    return d

def passed(r,rule):
    return bool(int(r[f'pass_{rule}'])==1)

def process_seed(seed):
    source=load_base(seed)
    assert len(source)==1312, f'Unexpected FULL size in seed {seed}: {len(source)}'
    assert (source['label'].astype(str).str.strip().str.lower()=='toxic').sum()==1150, 'Unexpected toxic-only size'
    folder=RUN_DIR/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    target=folder/'attempts.csv'
    records=[]; known={}
    if target.exists():
        if not RESUME_RUN: raise FileExistsError(f'Existing candidate log; enable explicit RESUME_RUN: {target}')
        old=pd.read_csv(target,keep_default_na=False)
        required_cache={'row_id','seed','attempt','X','Y','Y_sha256','label','band','STA','STA (indep.)','Sim','Flu','Joint','PPL','length_ratio','pass_original','pass_converted','fail_toxic','fail_semantic','fail_length','fail_flu_original','fail_flu_converted'}
        if not required_cache.issubset(old.columns):
            raise ValueError(f'Incomplete candidate log, missing {required_cache-set(old.columns)}')
        for r in old.to_dict('records'):
            idx=int(r['row_id']); a=int(r['attempt'])
            assert 0<=idx<len(source) and a in (1,2,3)
            assert int(r['seed'])==seed
            assert str(r['X'])==str(source.iloc[idx]['X'])
            assert str(r['label'])==str(source.iloc[idx]['label'])
            assert str(r['band'])==str(source.iloc[idx]['band'])
            assert str(r['Y_sha256'])==sha(r['Y'])
            if a==1: assert str(r['Y'])==str(source.iloc[idx]['Y'])
            # Re-score cached entries to detect stale metrics/scorer changes.
            fresh=assess(str(r['X']),str(r['Y']))
            for metric in ('STA','STA (indep.)','Sim','Flu','PPL','length_ratio','Joint'):
                if not np.isclose(float(r[metric]),float(fresh[metric]),atol=1e-5,rtol=1e-5):
                    raise ValueError(f'Cached scorer mismatch seed={seed} idx={idx} attempt={a} metric={metric}')
            for metric in ('pass_original','pass_converted','fail_toxic','fail_semantic','fail_length','fail_flu_original','fail_flu_converted'):
                if int(r[metric])!=int(fresh[metric]):
                    raise ValueError(f'Cached pass/fail mismatch seed={seed} idx={idx} attempt={a} metric={metric}')
            assert (idx,a) not in known
            known[idx,a]=r;records.append(r)
    model=None
    try:
        for idx,row in tqdm(source.iterrows(),total=len(source),desc=f'PostVal seed {seed}'):
            x=str(row['X']); seq=[]
            for attempt in (1,2,3):
                if attempt>1 and all(any(passed(r,rule) for r in seq)
                                      for rule in ('original','converted')):
                    break
                if (idx,attempt) in known:
                    r=known[idx,attempt]
                else:
                    if attempt==1:
                        y=str(row['Y']); temp=None
                    else:
                        if model is None:
                            ck=FLAT/f'seed_{seed}'/'flat_step_140'
                            model=AutoModelForSeq2SeqLMWithValueHead.from_pretrained(str(ck)).to(DEVICE).eval()
                        temp=REGEN_TEMPS[attempt-2]
                        y=generate_one(model,x,temp,seed,idx,attempt)
                    r={'row_id':idx,'seed':seed,'attempt':attempt,
                       'temperature':('beam4' if attempt==1 else temp),
                       'X':x,'Y':y,'Y_sha256':sha(y),
                       'label':str(row['label']),'band':str(row['band']),
                       **assess(x,y)}
                    known[idx,attempt]=r;records.append(r)
                    # Atomic replacement inside NEW folder only.
                    tmp=folder/'attempts.inprogress.tmp'
                    pd.DataFrame(records).sort_values(['row_id','attempt']).to_csv(tmp,index=False)
                    tmp.replace(target)
                seq.append(r)
            assert len(seq)==3 or all(any(passed(r,rule) for r in seq)
                                       for rule in ('original','converted'))
    finally:
        if model is not None: del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    all_att=pd.DataFrame(records).sort_values(['row_id','attempt']).reset_index(drop=True)
    assert all_att.groupby('row_id').size().ge(1).all()
    print(f'Seed {seed}: {len(source)} inputs, {len(all_att)} candidate rows -> {target}')
    return source,all_att

loaded={}
for seed in SEEDS:
    loaded[seed]=process_seed(seed)


PostVal seed 42:   0%|          | 0/1312 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

[transformers] T5ForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/seed_42/flat_step_140
Key                   | Status     |  | 
----------------------+------------+--+-
v_head.summary.bias   | UNEXPECTED |  | 
v_head.summary.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Seed 42: 1312 inputs, 3936 candidate rows -> /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/final_results_postval_fixed/run_20261014_133013/seed_42/attempts.csv


PostVal seed 123:   0%|          | 0/1312 [00:00<?, ?it/s]

...

Seed 123: 1312 inputs, 3936 candidate rows -> /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/final_results_postval_fixed/run_20261014_133013/seed_123/attempts.csv


PostVal seed 777:   0%|          | 0/1312 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

[transformers] T5ForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/seed_777/flat_step_140
Key                   | Status     |  | 
----------------------+------------+--+-
v_head.summary.bias   | UNEXPECTED |  | 
v_head.summary.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Seed 777: 1312 inputs, 3936 candidate rows -> /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/final_results_postval_fixed/run_20261014_133013/seed_777/attempts.csv


## PV-5 — Apply 2 thresholds x 2 fallbacks, compute edit metrics

In [ ]:
# Exactly as Config A: select first PASS per rule;
# if no PASS, ensure attempt 3 exists; fallback a=last, b=first.
def choose_variant(att,rule,fallback):
    output=[]
    for idx,g in att.groupby('row_id',sort=True):
        g=g.sort_values('attempt')
        good=g[g['pass_'+rule].astype(int)==1]
        if len(good):
            sel=good.iloc[0].copy(); n=int(sel['attempt']); invalid=False
        else:
            assert int(g.iloc[-1]['attempt'])==3,(idx,rule,len(g))
            sel=(g.iloc[-1] if fallback=='a' else g.iloc[0]).copy()
            n=3;invalid=True
        sel['n_attempts']=n
        sel['regenerated']=int(n>1)
        sel['failed_all']=int(invalid)
        sel['accepted']=int(not invalid)
        sel['manual_review']=int(invalid and fallback=='b')
        sel['accepted_attempt']=int(sel['attempt'])
        output.append(sel)
    return pd.DataFrame(output).reset_index(drop=True)

def levenshtein_tokens(a,b):
    p=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        q=[i]+[0]*len(b)
        for j,y in enumerate(b,1):
            q[j]=min(p[j]+1,q[j-1]+1,p[j-1]+(x!=y))
        p=q
    return p[-1]

def enrich(df):
    z=df.copy()
    x=z['X'].astype(str); y=z['Y'].astype(str)
    xt=x.str.lower().str.split(); yt=y.str.lower().str.split()
    lens=xt.map(len).clip(lower=1); leny=yt.map(len)
    ratio=leny/lens
    z['SLR (<0.70)']=(ratio<.7).astype(float)
    z['Mean Length Ratio']=ratio
    z['Truncation Rate (%)']=(leny<lens).astype(float)*100
    z['Clause Loss >30% (%)']=(ratio<.7).astype(float)*100
    z['Token Edit Rate']=[levenshtein_tokens(a,b)/max(len(a),1) for a,b in zip(xt,yt)]
    z['Unchanged Rate (%)']=(x.str.lower()==y.str.lower()).astype(float)*100
    z['PPL Median']=pd.to_numeric(z['PPL'],errors='coerce')
    return z

all_outputs={}
for seed,(base,att) in loaded.items():
    for rule in ('original','converted'):
        for fallback in ('a','b'):
            variant=f'{rule}_{fallback}'
            out=enrich(choose_variant(att,rule,fallback))
            target=RUN_DIR/f'seed_{seed}'/f'per_sample_{variant}.csv'
            if target.exists():
                if RESUME_RUN:
                    prev=pd.read_csv(target,keep_default_na=False)
                    assert len(prev)==len(out) and prev['Y'].astype(str).tolist()==out['Y'].astype(str).tolist(),f'Previously saved fallback mismatch: {target}'
                    all_outputs[seed,variant]=out
                    continue
                raise FileExistsError(target)
            out.to_csv(target,index=False)
            all_outputs[seed,variant]=out
print('Saved 4 variants per seed (4 fallback outcomes).')


Saved 4 variants per seed (4 fallback outcomes).


## PV-6 — Post-Val diagnostics FULL / TOXIC-ONLY

In [ ]:
# Diagnostics: replay each rule separately, not count all generated candidates
diagnostics=[]; reasons=[]; attempt_details=[]
for seed,(base,att) in loaded.items():
    for subset in ('FULL','TOXIC-ONLY'):
        ids=set(range(len(base))) if subset=='FULL' else set(base.index[
            base['label'].astype(str).str.lower()=='toxic'])
        a=att[att.row_id.isin(ids)]
        first=a[a.attempt==1]
        for rule in ('original','converted'):
            p=choose_variant(a,rule,'a')
            diagnostics.append({'seed':seed,'set':subset,'rule':rule,'n':len(first),
                'regeneration_rate':p.regenerated.mean(),
                'failed_all_rate':p.failed_all.mean(),
                'accepted_rate':p.accepted.mean(),
                'avg_attempts':p.n_attempts.mean(),
                'manual_review_rate_B':p.failed_all.mean()})
            reasons.append({'seed':seed,'set':subset,'rule':rule,'n':len(first),
                'Toxicity':first.fail_toxic.mean(),'Semantic':first.fail_semantic.mean(),
                'Fluency':first['fail_flu_'+rule].mean(),'Length':first.fail_length.mean(),
                'Too short':(first.length_ratio<.7).mean(),
                'Too long':(first.length_ratio>1.3).mean(),
                'Any':1-first['pass_'+rule].mean()})
            for step in (1,2,3):
                reached=[]
                for idx,g in a.groupby('row_id'):
                    g=g.sort_values('attempt')
                    prev=g[g.attempt<step]
                    if step==1 or (len(prev)==step-1 and (prev['pass_'+rule].astype(int)==0).all()):
                        hit=g[g.attempt==step]
                        if len(hit): reached.append(hit.iloc[0])
                if reached:
                    rr=pd.DataFrame(reached)
                    attempt_details.append({'seed':seed,'set':subset,'rule':rule,'attempt':step,
                        'n_reached':len(rr),'fail_length_rate':rr.fail_length.mean(),
                        'fail_short_rate':(rr.length_ratio<.7).mean(),
                        'fail_long_rate':(rr.length_ratio>1.3).mean()})
diag=pd.DataFrame(diagnostics); fail=pd.DataFrame(reasons)
master=RUN_DIR/'master_tables'
diag.to_csv(master/'regeneration_by_seed.csv',index=False)
fail.to_csv(master/'first_check_failures_by_seed.csv',index=False)
pd.DataFrame(attempt_details).to_csv(master/'length_diagnostics_by_attempt.csv',index=False)
print('Saved diagnostic CSVs.')


Saved diagnostic CSVs.


## PV-7 — Four groups of MASTER TABLES and MASTER_REPORT.md

In [ ]:
# Master Tables — one observation per seed; consistent with Config A reports.
# Each table aggregates THREE seed-level metrics, never six duplicated frames.
METRICS = [
    'STA','STA (indep.)','Sim','Flu','Joint',
    'SLR (<0.70)','Mean Length Ratio','Truncation Rate (%)',
    'Clause Loss >30% (%)','Token Edit Rate','Unchanged Rate (%)',
    'PPL Median','PPL IQR','% PPL [20,150]'
]
PERCENT_METRICS = {'Truncation Rate (%)','Clause Loss >30% (%)',
                   'Unchanged Rate (%)','% PPL [20,150]'}
PPL_METRICS = {'PPL Median','PPL IQR'}

def seed_summary(df):
    assert len(df), 'Empty group cannot be summarized'
    p = pd.to_numeric(df['PPL'], errors='raise')
    q1, q3 = p.quantile([.25,.75]).tolist()
    result = {}
    for m in METRICS:
        if m == 'PPL Median':
            result[m] = float(p.median())
        elif m == 'PPL IQR':
            result[m] = float(q3-q1)
        elif m == '% PPL [20,150]':
            result[m] = float(p.between(20,150,inclusive='both').mean()*100)
        else:
            result[m] = float(pd.to_numeric(df[m],errors='raise').mean())
    return result

def format_seed_stats(values, metric):
    v=np.asarray(values,dtype=float)
    assert len(v)==len(SEEDS) and np.isfinite(v).all(),(metric,v)
    mu,sd=v.mean(),v.std(ddof=1)
    if metric in PERCENT_METRICS:
        return f'{mu:.2f} ± {sd:.2f}%'
    if metric in PPL_METRICS:
        return f'{mu:.1f} ± {sd:.1f}'
    return f'{mu:.4f} ± {sd:.4f}'

def summarize_seed_frames(seed_frames):
    # Dict keyed by seed prevents accidental duplicated or missing seeds.
    assert set(seed_frames)==set(SEEDS),('Missing/extra seeds',seed_frames.keys())
    vals={seed:seed_summary(seed_frames[seed]) for seed in SEEDS}
    return {m:format_seed_stats([vals[seed][m] for seed in SEEDS],m)
            for m in METRICS}

# Verify saved candidate log and saved reference outputs are identical, row by row.
flat_by_seed={}
variant_by_seed={v:{} for v in ('original_a','original_b','converted_a','converted_b')}
for seed in SEEDS:
    base,att=loaded[seed]
    first=att[att['attempt'].astype(int)==1].sort_values('row_id')
    assert len(first)==len(base),(seed,len(first),len(base))
    assert first['row_id'].astype(int).tolist()==list(range(len(base)))
    assert first['X'].astype(str).tolist()==base['X'].astype(str).tolist()
    assert first['Y'].astype(str).tolist()==base['Y'].astype(str).tolist()
    flat_by_seed[seed]=enrich(first.reset_index(drop=True))
    for v in variant_by_seed:
        vdf=all_outputs[(seed,v)]
        assert len(vdf)==len(base),(seed,v)
        variant_by_seed[v][seed]=vdf

baseline_orig=pd.read_csv(SFT_CSV,keep_default_na=False)
assert {'X','Y','label','band'}.issubset(baseline_orig.columns)
assert baseline_orig['X'].astype(str).tolist()==loaded[SEEDS[0]][0]['X'].astype(str).tolist(), (
    'SFT reference input order does not match Flat C')
# Score actual SFT outputs, NEVER substitute Y=X.
baseline_path=RUN_DIR/'master_tables'/'sft_rescored.csv'
if baseline_path.exists() and RESUME_RUN:
    baseline=pd.read_csv(baseline_path,keep_default_na=False)
    assert len(baseline)==len(baseline_orig)
    assert baseline['X'].astype(str).tolist()==baseline_orig['X'].astype(str).tolist()
    assert baseline['Y'].astype(str).tolist()==baseline_orig['Y'].astype(str).tolist()
    if not {'STA','STA (indep.)','Sim','Flu','PPL','Joint'}.issubset(baseline.columns):
        raise ValueError('SFT cache missing required metrics; use new run.')
else:
    print('Scoring actual SFT Y (one reference, not an extra seed)...')
    rows=[]
    for r in tqdm(baseline_orig.itertuples(index=False),total=len(baseline_orig),desc='SFT reference'):
        rows.append(dict(X=str(r.X),Y=str(r.Y),label=r.label,band=r.band,
                         **assess(str(r.X),str(r.Y))))
    baseline=enrich(pd.DataFrame(rows))
    baseline.to_csv(baseline_path,index=False)

def toxic_only(df):
    return df[df['label'].astype(str).str.strip().str.lower()=='toxic'].copy()

# Baseline is the same reference for each seed; zero between-seed std is expected.
series={
    'Baseline (SFT)':{s:toxic_only(baseline) for s in SEEDS},
    'Flat PPO (before Post-Val)':{s:toxic_only(flat_by_seed[s]) for s in SEEDS},
}
variant_labels = {
    'original_a':'Flat + Post-Val (Original, Keep Attempt 3 [a])',
    'original_b':'Flat + Post-Val (Original, Keep Attempt 1 [b])',
    'converted_a':'Flat + Post-Val (Converted, Keep Attempt 3 [a])',
    'converted_b':'Flat + Post-Val (Converted, Keep Attempt 1 [b])',
}
for v,label in variant_labels.items():
    series[label]={s:toxic_only(variant_by_seed[v][s]) for s in SEEDS}

def build_table(named_groups,key_name):
    result=[]
    raw=[]
    for name,seed_frames in named_groups.items():
        st=summarize_seed_frames(seed_frames)
        result.append({key_name:name,**st})
        for seed in SEEDS:
            raw.append({key_name:name,'seed':seed,
                        'n':len(seed_frames[seed]),**seed_summary(seed_frames[seed])})
    return pd.DataFrame(result),pd.DataFrame(raw)

t1,t1_raw=build_table(series,'Method')
band_groups={}
for band in ['easy','medium','hard']:
    band_groups[band.capitalize()]={s:toxic_only(flat_by_seed[s]).loc[
        lambda f:f.band.astype(str).str.strip().str.lower()==band] for s in SEEDS}
band_groups['Overall']={s:toxic_only(flat_by_seed[s]) for s in SEEDS}
t2,t2_raw=build_table(band_groups,'Difficulty Band')
t3,t3_raw=build_table({k:series[k] for k in
                       ['Baseline (SFT)','Flat PPO (before Post-Val)']},'Phase')

def diagnostic_table(df,keys):
    out=[]
    assert set(df.seed)==set(SEEDS)
    for group,g in df.groupby(keys,sort=False):
        group=group if isinstance(group,tuple) else (group,)
        assert sorted(g['seed'].astype(int))==sorted(SEEDS),(keys,group)
        row=dict(zip(keys,group))
        for col in g.select_dtypes(include='number').columns:
            if col in ('seed','n'):continue
            v=g[col].to_numpy(dtype=float)
            mu,sd=v.mean(),v.std(ddof=1)
            row[col]=(f'{mu:.2f} ± {sd:.2f}' if col=='avg_attempts'
                      else f'{100*mu:.2f} ± {100*sd:.2f}%')
        out.append(row)
    return pd.DataFrame(out)

# Re-use the exact named diagnostics DataFrames generated in the preceding cell.
t4a=diagnostic_table(fail,['set','rule'])
t4b=diagnostic_table(diag,['set','rule'])
t4c,t4c_raw=build_table({k:series[k] for k in variant_labels.values()},'Method')

checks=[]
for seed in SEEDS:
    df=toxic_only(flat_by_seed[seed])
    a=loaded[seed][1]
    first=toxic_only(a[a.attempt==1])
    slr=float(df['SLR (<0.70)'].mean())
    short=float((first['length_ratio'].astype(float)<.70).mean())
    checks.append({'seed':seed,'n':len(df),'SLR Phase Flat':slr,
                   'Short-fail Attempt 1':short,'Match':bool(np.isclose(slr,short))})
tcheck=pd.DataFrame(checks)
assert tcheck['Match'].all(), 'Flat SLR does not match Attempt 1 length failures'

master=RUN_DIR/'master_tables'
table_specs=[
('GROUP 1: MAIN TABLE (PHASE FLAT TOXIC-ONLY)','group1_main_toxic_only.csv',t1),
('GROUP 2: BY BAND (PHASE FLAT TOXIC-ONLY)','group2_bands_toxic_only.csv',t2),
('GROUP 3: SFT vs FLAT (TOXIC-ONLY; NO CURRICULUM PHASES)','group3_sft_flat_toxic_only.csv',t3),
('GROUP 4A: TABLE G — FAILURE REASONS AT ATTEMPT 1 (FULL)','group4a_fail_reasons_full.csv',t4a[t4a['set']=='FULL']),
('GROUP 4A SUPPLEMENTARY: TABLE G — FAILURE REASONS AT ATTEMPT 1 (TOXIC-ONLY)','group4a_fail_reasons_toxic_only.csv',t4a[t4a['set']=='TOXIC-ONLY']),
('GROUP 4B: REGENERATION STATS (FULL)','group4b_regeneration_full.csv',t4b[t4b['set']=='FULL']),
('GROUP 4B SUPPLEMENTARY: REGENERATION STATS (TOXIC-ONLY)','group4b_regeneration_toxic_only.csv',t4b[t4b['set']=='TOXIC-ONLY']),
('CHECK SLR — FLAT vs ATTEMPT 1','slr_check.csv',tcheck),
('GROUP 4C: COMPARISON OF POST-VAL FALLBACK A/B','group4c_fallbacks.csv',t4c),
]
for fname,df in [
    ('group1_per_seed_numeric.csv',t1_raw),
    ('group2_per_seed_numeric.csv',t2_raw),
    ('group3_per_seed_numeric.csv',t3_raw),
    ('group4c_per_seed_numeric.csv',t4c_raw)
]:
    df.to_csv(master/fname,index=False)

report_lines=[
'# MASTER TABLES — FLAT PPO C Post-Validation',
f'- Run folder: {RUN_DIR}',
f'- Seeds: {SEEDS}; checkpoint: flat_step_140',
'- Attempt 1: stored beam output. Attempts 2–3: sampling at T=0.9/1.1.',
'- Original Flu > 0.70; converted Flu > {:.6f} (PPL < 139).'.format(FLU_CONV),
'- Four cases: each threshold × (a: keep attempt 3 / b: keep attempt 1 if all fail).',
'- Means and sample standard deviations are computed across exactly THREE seeds.',
'- FULL and TOXIC-ONLY are reported separately for rejection diagnostics.',
]
print('\n'+'='*72)
print('EXPORTED 4 GROUPS OF MASTER TABLES SUCCESSFULLY')
print('Results directory:', master)
print('='*72)
for title,fname,df in table_specs:
    df.to_csv(master/fname,index=False)
    print('\n>>>',title,'\n',df.to_string(index=False))
    report_lines.extend(['\n## '+title,'\n```text',df.to_string(index=False),'```'])
(master/'MASTER_REPORT.md').write_text('\n'.join(report_lines),encoding='utf-8')
print('\nSAVED CSV + MASTER_REPORT.MD')


Scoring actual SFT Y (one reference, not an extra seed)...


SFT reference:   0%|          | 0/1312 [00:00<?, ?it/s]


EXPORTED 4 GROUPS OF MASTER TABLES SUCCESSFULLY
Results directory: /content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1/final_results_postval_fixed/run_20261014_133013/master_tables

>>> GROUP 1: MAIN TABLE (PHASE FLAT TOXIC-ONLY) 
                                Method             STA    STA (indep.)             Sim             Flu           Joint     SLR (<0.70) Mean Length Ratio Truncation Rate (%) Clause Loss >30% (%) Token Edit Rate Unchanged Rate (%) PPL Median      PPL IQR % PPL [20,150]
                            Baseline (SFT) 0.8663 ± 0.0000 0.8946 ± 0.0000 0.9706 ± 0.0000 0.5499 ± 0.0000 0.7580 ± 0.0000 0.2574 ± 0.0000   0.8173 ± 0.0000     77.57 ± 0.00%   25.74 ± 0.00% 0.2004 ± 0.0000       16.26 ± 0.00%   62.5 ± 0.0   66.4 ± 0.0  84.09 ± 0.00%
                Flat PPO (before Post-Val) 0.8863 ± 0.0072 0.9144 ± 0.0094 0.9249 ± 0.0078 0.5378 ± 0.0033 0.7462 ± 0.0033 0.7188 ± 0.0443   0.5612 ± 0.0297     94.96 ± 0.63%   71.88 ± 4.43% 0.4609 ± 0.0330        2.06 ± 1.73%